# Foursquare Data Setup

Verifies the TIST 2015 dataset files are in place and previews the raw data.

In [11]:
import os, zipfile, glob

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
FS_DIR    = os.path.join(REPO_ROOT, "data", "foursquare")
os.makedirs(FS_DIR, exist_ok=True)

POI_FILE     = "dataset_TIST2015_POIs.txt"
CHECKIN_FILE = "dataset_TIST2015_Checkins.txt"

POI_PATH     = os.path.join(FS_DIR, POI_FILE)
CHECKIN_PATH = os.path.join(FS_DIR, CHECKIN_FILE)

print(FS_DIR)

/Users/ahmedabdelsalam/Desktop/PlanIt/planit/data/foursquare


In [8]:
for path, label in [(POI_PATH, POI_FILE), (CHECKIN_PATH, CHECKIN_FILE)]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {label:<40} {size_mb:.0f} MB  ✓")
    else:
        zips = glob.glob(os.path.join(FS_DIR, "*.zip"))
        if zips:
            print(f"  Extracting {label} from {os.path.basename(zips[0])} …")
            with zipfile.ZipFile(zips[0]) as z:
                match = next((n for n in z.namelist() if n.endswith(label)), None)
                if match:
                    z.extract(match, FS_DIR)
                    extracted = os.path.join(FS_DIR, match)
                    if extracted != path:
                        os.rename(extracted, path)
                    print(f"  Extracted → {path}")
                else:
                    print(f"  WARNING: {label} not found in zip")
        else:
            print(f"  MISSING: {label}")
            print(f"  Download dataset_TIST2015.zip and place it in {FS_DIR}")

  dataset_TIST2015_POIs.txt                232 MB  ✓
  dataset_TIST2015_Checkins.txt            1082 MB  ✓


In [9]:
import pandas as pd

pois = pd.read_csv(POI_PATH, sep="\t", header=None,
                   names=["venue_id", "latitude", "longitude", "category", "country_code"],
                   nrows=5)
print("POI sample:")
print(pois.to_string(index=False))

print()

checkins = pd.read_csv(CHECKIN_PATH, sep="\t", header=None,
                       names=["user_id", "venue_id", "utc_time", "tz_offset"],
                       nrows=5)
print("Check-in sample:")
print(checkins.to_string(index=False))

POI sample:
                venue_id  latitude  longitude          category country_code
3fd66200f964a52000e71ee3 40.733596 -74.003139         Jazz Club           US
3fd66200f964a52000e81ee3 40.758102 -73.975734               Gym           US
3fd66200f964a52000ea1ee3 40.732456 -74.003755 Indian Restaurant           US
3fd66200f964a52000ec1ee3 42.345907 -71.087001 Indian Restaurant           US
3fd66200f964a52000ee1ee3 39.933178 -75.159262    Sandwich Place           US

Check-in sample:
 user_id                 venue_id                       utc_time  tz_offset
   50756 4f5e3a72e4b053fd6a4313f6 Tue Apr 03 18:00:06 +0000 2012        240
  190571 4b4b87b5f964a5204a9f26e3 Tue Apr 03 18:00:07 +0000 2012        180
  221021 4a85b1b3f964a520eefe1fe3 Tue Apr 03 18:00:08 +0000 2012       -240
   66981 4b4606f2f964a520751426e3 Tue Apr 03 18:00:08 +0000 2012       -300
   21010 4c2b4e8a9a559c74832f0de2 Tue Apr 03 18:00:09 +0000 2012        240


In [10]:
# quick stats on category distribution
all_pois = pd.read_csv(POI_PATH, sep="\t", header=None,
                       names=["venue_id", "latitude", "longitude", "category", "country_code"])

print(f"Total venues: {len(all_pois):,}")
print(f"Unique categories: {all_pois['category'].nunique():,}")
print(f"Countries: {all_pois['country_code'].nunique()}")
print()
print("Top 15 categories:")
print(all_pois["category"].value_counts().head(15).to_string())

Total venues: 3,680,126
Unique categories: 429
Countries: 77

Top 15 categories:
category
Home (private)                              322116
Office                                      116941
Residential Building (Apartment / Condo)    112929
Building                                     87975
Caf                                        78053
Road                                         53817
Bank                                         53411
Restaurant                                   50169
Bar                                          46035
Salon / Barbershop                           45462
Coffee Shop                                  42008
Automotive Shop                              40801
Gas Station / Garage                         39762
Grocery Store                                39381
Hotel                                        37063
